# 🔋 Global Lithium Mining Companies Analysis
## Comprehensive Financial, Technical, Geospatial & Operational Analysis

**Analysis Date:** 2026-02-23  
**Coverage:** 50+ lithium mining companies across global exchanges  
**Scope:** NYSE/NASDAQ, TSX/TSX-V, ASX, and international listings

---

### Analysis Dimensions:
- 📊 **Fundamental Financial Analysis**: Balance sheets, cash flow, profitability metrics
- 💰 **Valuation**: DCF models, NAV analysis, valuation multiples
- 📈 **Technical Analysis**: RSI, MACD, Bollinger Bands, momentum indicators
- 🗺️ **Geospatial Visualization**: Global mine mapping with production capacity
- 📉 **Cost Curve Analysis**: Production cost positioning
- 📑 **SEC Filings**: Reserve/resource extraction from 10-K reports
- 🎯 **Composite Scoring**: Multi-factor investment ranking system

---

## SECTION 0: Setup & Configuration

In [1]:
# Package Installation
!pip install -q yfinance pandas numpy plotly scipy geopy sec-edgar-downloader beautifulsoup4 tqdm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Core Imports
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# Standard Library
from datetime import datetime, timedelta
import time
import json
import pickle
import re
import warnings
warnings.filterwarnings('ignore')

# Analysis Libraries
from scipy import stats
from tqdm import tqdm

# Geospatial
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

# SEC Filings
from sec_edgar_downloader import Downloader
from bs4 import BeautifulSoup

# Configuration
pio.templates.default = "plotly_dark"
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ All packages imported successfully!")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ All packages imported successfully!
Analysis Date: 2026-02-24 08:14:18


### Company Universe Definition

Comprehensive list of lithium mining companies across all development stages and exchanges.

In [3]:
# Global Lithium Mining Companies Universe
# Type: Integrated (mining+processing), Producer (mining), Developer (pre-production), Explorer (early-stage)

LITHIUM_MINERS = {
    # === NORTH AMERICA - INTEGRATED & PRODUCERS ===
    'ALB': {'name': 'Albemarle Corporation', 'region': 'North America', 'type': 'Integrated', 
            'currency': 'USD', 'country': 'USA', 'exchange': 'NYSE',
            'description': 'Global lithium leader, brine + hard rock operations'},
    
    'LTHM': {'name': 'Livent Corporation', 'region': 'North America', 'type': 'Integrated',
             'currency': 'USD', 'country': 'USA', 'exchange': 'NYSE',
             'description': 'Lithium chemicals producer, Argentina operations'},
    
    'LAC': {'name': 'Lithium Americas Corp', 'region': 'North America', 'type': 'Developer',
            'currency': 'USD', 'country': 'Canada', 'exchange': 'NYSE',
            'description': 'Thacker Pass (Nevada) and Argentina projects'},
    
    'LAC.TO': {'name': 'Lithium Americas Corp', 'region': 'North America', 'type': 'Developer',
               'currency': 'CAD', 'country': 'Canada', 'exchange': 'TSX',
               'description': 'TSX listing of Lithium Americas'},
    
    'LTBR': {'name': 'Lithium Americas (Argentina)', 'region': 'Latin America', 'type': 'Developer',
             'currency': 'USD', 'country': 'Argentina', 'exchange': 'NYSE',
             'description': 'Cauchari-Olaroz project'},
    
    'PLL': {'name': 'Piedmont Lithium', 'region': 'North America', 'type': 'Developer',
            'currency': 'USD', 'country': 'USA', 'exchange': 'NASDAQ',
            'description': 'Carolina Lithium project + Ghana'},
    
    'IONR': {'name': 'ioneer Ltd', 'region': 'North America', 'type': 'Developer',
             'currency': 'USD', 'country': 'Australia', 'exchange': 'NASDAQ',
             'description': 'Rhyolite Ridge lithium-boron project, Nevada'},
    
    'CYN.V': {'name': 'Cypress Development Corp', 'region': 'North America', 'type': 'Explorer',
              'currency': 'CAD', 'country': 'Canada', 'exchange': 'TSX-V',
              'description': 'Clayton Valley lithium project, Nevada'},
    
    'LITM': {'name': 'Snow Lake Resources', 'region': 'North America', 'type': 'Explorer',
             'currency': 'USD', 'country': 'Canada', 'exchange': 'NASDAQ',
             'description': 'Manitoba lithium project'},
    
    'FRHC': {'name': 'Freedom Holding Corp', 'region': 'North America', 'type': 'Explorer',
             'currency': 'USD', 'country': 'USA', 'exchange': 'NASDAQ',
             'description': 'Nevada lithium projects'},
    
    'LIACF': {'name': 'Lithium Ionic Corp', 'region': 'North America', 'type': 'Explorer',
              'currency': 'USD', 'country': 'Canada', 'exchange': 'OTC',
              'description': 'Brazil lithium projects'},
    
    # === SOUTH AMERICA - MAJOR PRODUCERS ===
    'SQM': {'name': 'Sociedad Química y Minera', 'region': 'Latin America', 'type': 'Integrated',
            'currency': 'USD', 'country': 'Chile', 'exchange': 'NYSE',
            'description': 'Chilean lithium giant, Atacama brine operations'},
    
    'LTHM.BA': {'name': 'Livent - Buenos Aires', 'region': 'Latin America', 'type': 'Producer',
                'currency': 'ARS', 'country': 'Argentina', 'exchange': 'BYMA',
                'description': 'Argentina operations'},
    
    'PLLAF': {'name': 'Pilbara Minerals - ADR', 'region': 'Asia-Pacific', 'type': 'Producer',
              'currency': 'USD', 'country': 'Australia', 'exchange': 'OTC',
              'description': 'ADR of Pilbara Minerals'},
    
    # === AUSTRALIA - MAJOR HARD ROCK PRODUCERS ===
    'PLS.AX': {'name': 'Pilbara Minerals Limited', 'region': 'Asia-Pacific', 'type': 'Producer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Pilgangoora lithium operation, largest ASX producer'},
    
    'AKE.AX': {'name': 'Allkem Limited', 'region': 'Asia-Pacific', 'type': 'Integrated',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Olaroz (Argentina) + Mt Cattlin (Australia)'},
    
    'MIN.AX': {'name': 'Mineral Resources Limited', 'region': 'Asia-Pacific', 'type': 'Diversified',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Mt Marion, Wodgina lithium projects + iron ore'},
    
    'IGO.AX': {'name': 'IGO Limited', 'region': 'Asia-Pacific', 'type': 'Diversified',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Greenbushes JV (25%), nickel operations'},
    
    'LTR.AX': {'name': 'Liontown Resources', 'region': 'Asia-Pacific', 'type': 'Developer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Kathleen Valley lithium project, WA'},
    
    'LKE.AX': {'name': 'Lake Resources NL', 'region': 'Asia-Pacific', 'type': 'Developer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Kachi brine project, Argentina'},
    
    'CXO.AX': {'name': 'Core Lithium Limited', 'region': 'Asia-Pacific', 'type': 'Producer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Finniss lithium project, Northern Territory'},
    
    'ESS.AX': {'name': 'Essential Metals Limited', 'region': 'Asia-Pacific', 'type': 'Developer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Pioneer Dome lithium project, WA'},
    
    'LPD.AX': {'name': 'Lepidico Limited', 'region': 'Asia-Pacific', 'type': 'Developer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Lepidolite processing technology'},
    
    'AVZ.AX': {'name': 'AVZ Minerals Limited', 'region': 'Africa', 'type': 'Developer',
               'currency': 'AUD', 'country': 'DRC', 'exchange': 'ASX',
               'description': 'Manono lithium-tin project, DRC'},
    
    'LRS.AX': {'name': 'Latin Resources Limited', 'region': 'Latin America', 'type': 'Explorer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Salinas lithium project, Brazil'},
    
    'AGY.AX': {'name': 'Argosy Minerals Limited', 'region': 'Latin America', 'type': 'Developer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Rincon lithium project, Argentina'},
    
    'GLN.AX': {'name': 'Galan Lithium Limited', 'region': 'Latin America', 'type': 'Developer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Hombre Muerto West, Argentina'},
    
    'DLC.AX': {'name': 'Delta Lithium Limited', 'region': 'Asia-Pacific', 'type': 'Explorer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Mt Ida lithium project, WA'},
    
    'AZL.AX': {'name': 'Arizona Lithium Limited', 'region': 'North America', 'type': 'Explorer',
               'currency': 'AUD', 'country': 'USA', 'exchange': 'ASX',
               'description': 'Big Sandy lithium project, Arizona'},
    
    'LEL.AX': {'name': 'Lithium Energy Limited', 'region': 'Asia-Pacific', 'type': 'Explorer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Burke lithium project, Queensland'},
    
    'NVX.AX': {'name': 'Novonix Limited', 'region': 'Asia-Pacific', 'type': 'Integrated',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Battery anode materials, synthetic graphite'},
    
    'SYA.AX': {'name': 'Sayona Mining Limited', 'region': 'North America', 'type': 'Developer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Quebec lithium projects, Canada'},
    
    'VUL.AX': {'name': 'Vulcan Energy Resources', 'region': 'Europe', 'type': 'Developer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Zero-carbon lithium from geothermal, Germany'},
    
    'WC8.AX': {'name': 'Wildcat Resources Limited', 'region': 'Asia-Pacific', 'type': 'Explorer',
               'currency': 'AUD', 'country': 'Australia', 'exchange': 'ASX',
               'description': 'Tabba Tabba lithium project, WA'},
    
    # === CANADA - TSX/TSX-V ===
    'LI.V': {'name': 'Lithium Ionic Corp', 'region': 'Latin America', 'type': 'Explorer',
             'currency': 'CAD', 'country': 'Canada', 'exchange': 'TSX-V',
             'description': 'Bandeira lithium project, Brazil'},
    
    'PMET.V': {'name': 'Patriot Battery Metals', 'region': 'North America', 'type': 'Explorer',
               'currency': 'CAD', 'country': 'Canada', 'exchange': 'TSX-V',
               'description': 'Corvette lithium project, James Bay, Quebec'},
    
    'CRE.V': {'name': 'Critical Elements Lithium Corp', 'region': 'North America', 'type': 'Developer',
              'currency': 'CAD', 'country': 'Canada', 'exchange': 'TSX-V',
              'description': 'Rose lithium-tantalum project, Quebec'},
    
    'FLPC.V': {'name': 'Frontier Lithium', 'region': 'North America', 'type': 'Developer',
               'currency': 'CAD', 'country': 'Canada', 'exchange': 'TSX-V',
               'description': 'PAK lithium project, Ontario'},
    
    'LITM.V': {'name': 'Lithium Chile Inc', 'region': 'Latin America', 'type': 'Explorer',
               'currency': 'CAD', 'country': 'Canada', 'exchange': 'TSX-V',
               'description': 'Chilean lithium brine projects'},
    
    'LI.TO': {'name': 'Li-FT Power Ltd', 'region': 'North America', 'type': 'Explorer',
              'currency': 'CAD', 'country': 'Canada', 'exchange': 'TSX-V',
              'description': 'Yellowknife lithium project, NWT'},
    
    'SGML.V': {'name': 'Sigma Lithium Corporation', 'region': 'Latin America', 'type': 'Producer',
               'currency': 'CAD', 'country': 'Canada', 'exchange': 'TSX-V',
               'description': 'Grota do Cirilo, Brazil - in production'},
    
    'SGML': {'name': 'Sigma Lithium Corporation', 'region': 'Latin America', 'type': 'Producer',
             'currency': 'USD', 'country': 'Canada', 'exchange': 'NASDAQ',
             'description': 'NASDAQ listing of Sigma Lithium'},
    
    'E3M.V': {'name': 'E3 Lithium Ltd', 'region': 'North America', 'type': 'Developer',
              'currency': 'CAD', 'country': 'Canada', 'exchange': 'TSX-V',
              'description': 'Direct lithium extraction, Alberta'},
    
    'PNXLF': {'name': 'Power Nickel Inc', 'region': 'North America', 'type': 'Explorer',
              'currency': 'USD', 'country': 'Canada', 'exchange': 'OTC',
              'description': 'Chile lithium brine projects'},
    
    # === CHINA - MAJOR INTEGRATED PRODUCERS ===
    '002460.SZ': {'name': 'Ganfeng Lithium', 'region': 'Asia-Pacific', 'type': 'Integrated',
                  'currency': 'CNY', 'country': 'China', 'exchange': 'Shenzhen',
                  'description': 'Worlds largest lithium metals producer'},
    
    '01772.HK': {'name': 'Ganfeng Lithium - HK', 'region': 'Asia-Pacific', 'type': 'Integrated',
                 'currency': 'HKD', 'country': 'China', 'exchange': 'HKEX',
                 'description': 'Hong Kong listing of Ganfeng'},
    
    '002074.SZ': {'name': 'Tianqi Lithium', 'region': 'Asia-Pacific', 'type': 'Integrated',
                  'currency': 'CNY', 'country': 'China', 'exchange': 'Shenzhen',
                  'description': '51% Greenbushes, SQM stake'},
    
    '09696.HK': {'name': 'Tianqi Lithium - HK', 'region': 'Asia-Pacific', 'type': 'Integrated',
                 'currency': 'HKD', 'country': 'China', 'exchange': 'HKEX',
                 'description': 'Hong Kong listing of Tianqi'},
    
    # === EUROPE ===
    'RIO.L': {'name': 'Rio Tinto - Lithium Division', 'region': 'Europe', 'type': 'Diversified',
              'currency': 'GBP', 'country': 'UK', 'exchange': 'LSE',
              'description': 'Rincon (Argentina) + Jadar (Serbia) projects'},
    
    'SAVNF': {'name': 'Savannah Resources', 'region': 'Europe', 'type': 'Developer',
              'currency': 'USD', 'country': 'UK', 'exchange': 'OTC',
              'description': 'Barroso lithium project, Portugal'},
    
    'BSSLF': {'name': 'Bacanora Lithium', 'region': 'North America', 'type': 'Developer',
              'currency': 'USD', 'country': 'UK', 'exchange': 'OTC',
              'description': 'Sonora lithium project, Mexico'},
    
    # === OTHER DIVERSIFIED MAJORS ===
    'RIO': {'name': 'Rio Tinto', 'region': 'Global', 'type': 'Diversified',
            'currency': 'USD', 'country': 'UK/Australia', 'exchange': 'NYSE',
            'description': 'Major miner with lithium division'},
}

print(f"✅ Company universe defined: {len(LITHIUM_MINERS)} companies")
print(f"\nBreakdown by type:")
type_counts = pd.Series([v['type'] for v in LITHIUM_MINERS.values()]).value_counts()
print(type_counts)
print(f"\nBreakdown by region:")
region_counts = pd.Series([v['region'] for v in LITHIUM_MINERS.values()]).value_counts()
print(region_counts)

✅ Company universe defined: 52 companies

Breakdown by type:
Developer      19
Explorer       14
Integrated      9
Producer        6
Diversified     4
Name: count, dtype: int64

Breakdown by region:
North America    19
Asia-Pacific     18
Latin America    10
Europe            3
Africa            1
Global            1
Name: count, dtype: int64


### Helper Functions

Reusable utility functions for data extraction and analysis.

In [4]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_get(d, key, default=np.nan):
    """Safely extract a value from dict."""
    val = d.get(key, default)
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return default
    return val

def safe_stmt_get(stmt, row_name, col_idx=0):
    """Safely extract a value from a financial statement DataFrame."""
    try:
        if stmt is None or stmt.empty:
            return np.nan
        if row_name in stmt.index:
            val = stmt.loc[row_name].iloc[col_idx] if hasattr(stmt.loc[row_name], 'iloc') else stmt.loc[row_name]
            return val if not pd.isna(val) else np.nan
        return np.nan
    except:
        return np.nan

def compute_growth(stmt, row_name, periods=1):
    """Calculate YoY growth rate from financial statement."""
    try:
        if stmt is None or stmt.empty or row_name not in stmt.index:
            return np.nan
        if len(stmt.columns) < periods + 1:
            return np.nan
        recent = stmt.loc[row_name].iloc[0]
        prior = stmt.loc[row_name].iloc[periods]
        if pd.notna(recent) and pd.notna(prior) and prior != 0:
            return ((recent / prior) - 1) * 100
        return np.nan
    except:
        return np.nan

def format_currency(value, currency='USD'):
    """Format large numbers with B/M/K suffixes."""
    if pd.isna(value):
        return 'N/A'
    abs_val = abs(value)
    if abs_val >= 1e9:
        return f"{currency} {value/1e9:.2f}B"
    elif abs_val >= 1e6:
        return f"{currency} {value/1e6:.2f}M"
    elif abs_val >= 1e3:
        return f"{currency} {value/1e3:.2f}K"
    else:
        return f"{currency} {value:.2f}"

def percentile_rank(series, higher_is_better=True):
    """Rank values on 0-100 percentile scale."""
    if higher_is_better:
        return series.rank(pct=True) * 100
    else:
        return (1 - series.rank(pct=True)) * 100

def geocode_location(location_string, max_retries=3):
    """Convert location string to lat/lon coordinates."""
    geolocator = Nominatim(user_agent="lithium_miner_analysis")
    for attempt in range(max_retries):
        try:
            location = geolocator.geocode(location_string)
            if location:
                return location.latitude, location.longitude
            time.sleep(1)
        except GeocoderTimedOut:
            if attempt < max_retries - 1:
                time.sleep(2)
            continue
    return None, None

print("✅ Helper functions defined successfully!")

✅ Helper functions defined successfully!


## SECTION 1: Data Collection

Fetch comprehensive data for all lithium mining companies with retry logic and caching.

In [5]:
# Data fetching function with retry logic
def fetch_lithium_miner_data(ticker, meta, max_retries=3):
    """Fetch comprehensive financial data for a single lithium mining company."""
    for attempt in range(max_retries):
        try:
            t = yf.Ticker(ticker)
            info = t.info
            
            # Validate we got real data
            mc = info.get("marketCap", 0)
            if not mc or mc == 0:
                if attempt < max_retries - 1:
                    time.sleep(2)
                    continue
                return None
            
            # Pull financial statements
            income = t.income_stmt
            balance = t.balance_sheet
            cashflow = t.cashflow
            
            # 5-year price history
            hist = t.history(period="5y")
            
            return {
                "ticker": ticker,
                "name": meta["name"],
                "region": meta["region"],
                "type": meta["type"],
                "currency": meta["currency"],
                "country": meta["country"],
                "exchange": meta["exchange"],
                "info": info,
                "income_stmt": income,
                "balance_sheet": balance,
                "cashflow": cashflow,
                "price_history": hist,
                "data_quality": 100 if (not income.empty and not balance.empty) else 50
            }
        except Exception as e:
            print(f"  ⚠️  Error fetching {ticker} (attempt {attempt+1}): {str(e)[:100]}")
            if attempt < max_retries - 1:
                time.sleep(3)
            else:
                return None
    return None

print("✅ Data fetching function defined!")

✅ Data fetching function defined!


In [6]:
# Batch data collection with caching
CACHE_FILE = "lithium_miners_data.pkl"

def collect_all_data(force_refresh=False):
    """Collect data for all companies with progress tracking."""
    
    # Check for cached data
    if not force_refresh and os.path.exists(CACHE_FILE):
        print(f"📁 Loading cached data from {CACHE_FILE}...")
        with open(CACHE_FILE, "rb") as f:
            return pickle.load(f)
    
    print(f"🔄 Fetching data for {len(LITHIUM_MINERS)} companies...")
    
    all_data = {}
    failed = []
    
    for ticker, meta in tqdm(LITHIUM_MINERS.items(), desc="Fetching"):
        data = fetch_lithium_miner_data(ticker, meta)
        if data:
            all_data[ticker] = data
        else:
            failed.append(ticker)
        time.sleep(0.5)  # Rate limiting
    
    # Save to cache
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(all_data, f)
    
    print(f"✅ Data collection complete!")
    print(f"   - Successfully fetched: {len(all_data)} companies")
    print(f"   - Failed: {len(failed)} companies")
    if failed:
        print(f"   - Failed tickers: {', '.join(failed[:10])}")
    
    return all_data

# Execute data collection
import os
lithium_data = collect_all_data(force_refresh=False)
print(f"📊 Ready to analyze {len(lithium_data)} companies!")

📁 Loading cached data from lithium_miners_data.pkl...
📊 Ready to analyze 29 companies!


## SECTION 2: Fundamental Financial Analysis

Extract key financial metrics from balance sheets, income statements, and cash flow statements.

In [7]:
# Extract comprehensive financial metrics
def extract_mining_metrics(ticker, data):
    """Extract comprehensive metrics from financial statements."""
    info = data["info"]
    income = data["income_stmt"]
    balance = data["balance_sheet"]
    cashflow = data["cashflow"]
    
    metrics = {
        "ticker": ticker,
        "name": data["name"],
        "type": data["type"],
        "region": data["region"],
    }
    
    # Market Data
    metrics["market_cap"] = safe_get(info, "marketCap")
    metrics["enterprise_value"] = safe_get(info, "enterpriseValue")
    metrics["current_price"] = safe_get(info, "currentPrice")
    metrics["52w_high"] = safe_get(info, "fiftyTwoWeekHigh")
    metrics["52w_low"] = safe_get(info, "fiftyTwoWeekLow")
    
    # Revenue & Profitability
    metrics["revenue"] = safe_stmt_get(income, "Total Revenue")
    metrics["gross_profit"] = safe_stmt_get(income, "Gross Profit")
    metrics["operating_income"] = safe_stmt_get(income, "Operating Income")
    metrics["net_income"] = safe_stmt_get(income, "Net Income")
    metrics["ebitda"] = safe_stmt_get(income, "EBITDA")
    
    # Calculate margins
    if pd.notna(metrics["revenue"]) and metrics["revenue"] != 0:
        metrics["gross_margin"] = (metrics["gross_profit"] / metrics["revenue"]) * 100 if pd.notna(metrics["gross_profit"]) else np.nan
        metrics["operating_margin"] = (metrics["operating_income"] / metrics["revenue"]) * 100 if pd.notna(metrics["operating_income"]) else np.nan
        metrics["net_margin"] = (metrics["net_income"] / metrics["revenue"]) * 100 if pd.notna(metrics["net_income"]) else np.nan
    else:
        metrics["gross_margin"] = metrics["operating_margin"] = metrics["net_margin"] = np.nan
    
    # Balance Sheet
    metrics["total_assets"] = safe_stmt_get(balance, "Total Assets")
    metrics["current_assets"] = safe_stmt_get(balance, "Current Assets")
    metrics["cash"] = safe_stmt_get(balance, "Cash And Cash Equivalents")
    metrics["total_debt"] = safe_stmt_get(balance, "Total Debt")
    metrics["current_liabilities"] = safe_stmt_get(balance, "Current Liabilities")
    metrics["stockholders_equity"] = safe_stmt_get(balance, "Stockholders Equity")
    
    # Liquidity Ratios
    if pd.notna(metrics["current_liabilities"]) and metrics["current_liabilities"] != 0:
        metrics["current_ratio"] = metrics["current_assets"] / metrics["current_liabilities"] if pd.notna(metrics["current_assets"]) else np.nan
    else:
        metrics["current_ratio"] = np.nan
    
    # Leverage Ratios
    if pd.notna(metrics["stockholders_equity"]) and metrics["stockholders_equity"] != 0:
        metrics["debt_to_equity"] = metrics["total_debt"] / metrics["stockholders_equity"] if pd.notna(metrics["total_debt"]) else np.nan
    else:
        metrics["debt_to_equity"] = np.nan
    
    metrics["net_debt"] = (metrics["total_debt"] - metrics["cash"]) if pd.notna(metrics["total_debt"]) and pd.notna(metrics["cash"]) else np.nan
    
    # Cash Flow
    metrics["operating_cf"] = safe_stmt_get(cashflow, "Operating Cash Flow")
    metrics["capex"] = safe_stmt_get(cashflow, "Capital Expenditure")
    metrics["free_cash_flow"] = (metrics["operating_cf"] + metrics["capex"]) if pd.notna(metrics["operating_cf"]) and pd.notna(metrics["capex"]) else np.nan
    
    # Mining-specific metrics
    if pd.notna(metrics["revenue"]) and metrics["revenue"] != 0:
        metrics["capex_intensity"] = abs(metrics["capex"] / metrics["revenue"]) * 100 if pd.notna(metrics["capex"]) else np.nan
    else:
        metrics["capex_intensity"] = np.nan
    
    # Valuation Ratios
    metrics["pe_ratio"] = safe_get(info, "trailingPE")
    metrics["pb_ratio"] = safe_get(info, "priceToBook")
    metrics["ps_ratio"] = safe_get(info, "priceToSalesTrailing12Months")
    
    if pd.notna(metrics["enterprise_value"]) and pd.notna(metrics["ebitda"]) and metrics["ebitda"] != 0:
        metrics["ev_ebitda"] = metrics["enterprise_value"] / metrics["ebitda"]
    else:
        metrics["ev_ebitda"] = np.nan
    
    # Returns
    metrics["roe"] = safe_get(info, "returnOnEquity") * 100 if safe_get(info, "returnOnEquity") else np.nan
    metrics["roa"] = safe_get(info, "returnOnAssets") * 100 if safe_get(info, "returnOnAssets") else np.nan
    
    # Growth
    metrics["revenue_growth"] = compute_growth(income, "Total Revenue")
    
    return metrics

print("✅ Metrics extraction function defined!")

✅ Metrics extraction function defined!


In [8]:
# Extract metrics for all companies
print("📊 Extracting financial metrics for all companies...\n")

metrics_list = []
for ticker, data in tqdm(lithium_data.items(), desc="Extracting metrics"):
    metrics = extract_mining_metrics(ticker, data)
    metrics_list.append(metrics)

df_metrics = pd.DataFrame(metrics_list)
df_metrics.set_index("ticker", inplace=True)

print(f"✅ Metrics extracted for {len(df_metrics)} companies!")
print(f"\nDataFrame shape: {df_metrics.shape}")
print(f"\nSample metrics:")
df_metrics[["name", "market_cap", "revenue", "net_margin", "debt_to_equity", "pe_ratio"]].head(10)

📊 Extracting financial metrics for all companies...



Extracting metrics: 100%|██████████| 29/29 [00:00<00:00, 464.48it/s]

✅ Metrics extracted for 29 companies!

DataFrame shape: (29, 36)

Sample metrics:


,name,market_cap,revenue,net_margin,debt_to_equity,pe_ratio
ticker,,,,,,
ALB,Albemarle Corporation,20920238080,5.142733e+09,-9.929117,0.345801,NaN
LAC,Lithium Americas Corp,1386941568,0.000000e+00,NaN,0.035649,NaN
LAC.TO,Lithium Americas Corp,1899836800,0.000000e+00,NaN,0.035649,NaN
LTBR,Lithium Americas (Argentina),403994272,0.000000e+00,NaN,NaN,NaN
IONR,ioneer Ltd,311167552,0.000000e+00,NaN,0.001620,NaN
LITM,Snow Lake Resources,53718284,0.000000e+00,NaN,0.001220,NaN
FRHC,Freedom Holding Corp,7205797888,2.033455e+09,4.162866,1.608724,5888.0
SQM,Sociedad Química y Minera,20871555072,4.528761e+09,-8.928689,0.933706,NaN
PLS.AX,Pilbara Minerals Limited,15202500608,8.902070e+08,-21.991065,0.193184,NaN


## SECTION 2: Fundamental Financial Analysis

In [9]:
def extract_mining_metrics(ticker, data):
    info = data["info"]
    metrics = {"ticker": ticker, "name": data["name"], "type": data["type"]}
    metrics["market_cap"] = safe_get(info, "marketCap")
    metrics["revenue"] = safe_stmt_get(data["income_stmt"], "Total Revenue")
    metrics["net_margin"] = 10.0  # Placeholder
    return metrics
print("Metrics function defined\!")

Metrics function defined\!


## SECTION 2: Fundamental Financial Analysis

Extract key financial metrics from balance sheets, income statements, and cash flow statements.

In [10]:
# Extract comprehensive financial metrics
def extract_mining_metrics(ticker, data):
    """Extract comprehensive metrics from financial statements."""
    info = data["info"]
    income = data["income_stmt"]
    balance = data["balance_sheet"]
    cashflow = data["cashflow"]

    metrics = {
        "ticker": ticker,
        "name": data["name"],
        "type": data["type"],
        "region": data["region"],
    }

    # Market Data
    metrics["market_cap"] = safe_get(info, "marketCap")
    metrics["enterprise_value"] = safe_get(info, "enterpriseValue")
    metrics["current_price"] = safe_get(info, "currentPrice")

    # Revenue & Profitability
    metrics["revenue"] = safe_stmt_get(income, "Total Revenue")
    metrics["gross_profit"] = safe_stmt_get(income, "Gross Profit")
    metrics["operating_income"] = safe_stmt_get(income, "Operating Income")
    metrics["net_income"] = safe_stmt_get(income, "Net Income")
    metrics["ebitda"] = safe_stmt_get(income, "EBITDA")

    # Calculate margins
    if pd.notna(metrics["revenue"]) and metrics["revenue"] != 0:
        metrics["gross_margin"] = (metrics["gross_profit"] / metrics["revenue"]) * 100 if pd.notna(metrics["gross_profit"]) else np.nan
        metrics["operating_margin"] = (metrics["operating_income"] / metrics["revenue"]) * 100 if pd.notna(metrics["operating_income"]) else np.nan
        metrics["net_margin"] = (metrics["net_income"] / metrics["revenue"]) * 100 if pd.notna(metrics["net_income"]) else np.nan
    else:
        metrics["gross_margin"] = metrics["operating_margin"] = metrics["net_margin"] = np.nan

    # Balance Sheet
    metrics["total_assets"] = safe_stmt_get(balance, "Total Assets")
    metrics["current_assets"] = safe_stmt_get(balance, "Current Assets")
    metrics["cash"] = safe_stmt_get(balance, "Cash And Cash Equivalents")
    metrics["total_debt"] = safe_stmt_get(balance, "Total Debt")
    metrics["current_liabilities"] = safe_stmt_get(balance, "Current Liabilities")
    metrics["stockholders_equity"] = safe_stmt_get(balance, "Stockholders Equity")

    # Ratios
    if pd.notna(metrics["current_liabilities"]) and metrics["current_liabilities"] != 0:
        metrics["current_ratio"] = metrics["current_assets"] / metrics["current_liabilities"] if pd.notna(metrics["current_assets"]) else np.nan
    else:
        metrics["current_ratio"] = np.nan

    if pd.notna(metrics["stockholders_equity"]) and metrics["stockholders_equity"] != 0:
        metrics["debt_to_equity"] = metrics["total_debt"] / metrics["stockholders_equity"] if pd.notna(metrics["total_debt"]) else np.nan
    else:
        metrics["debt_to_equity"] = np.nan

    # Cash Flow
    metrics["operating_cf"] = safe_stmt_get(cashflow, "Operating Cash Flow")
    metrics["capex"] = safe_stmt_get(cashflow, "Capital Expenditure")
    metrics["free_cash_flow"] = (metrics["operating_cf"] + metrics["capex"]) if pd.notna(metrics["operating_cf"]) and pd.notna(metrics["capex"]) else np.nan

    # Valuation
    metrics["pe_ratio"] = safe_get(info, "trailingPE")
    metrics["pb_ratio"] = safe_get(info, "priceToBook")
    metrics["ev_ebitda"] = metrics["enterprise_value"] / metrics["ebitda"] if pd.notna(metrics["enterprise_value"]) and pd.notna(metrics["ebitda"]) and metrics["ebitda"] != 0 else np.nan

    # Returns
    metrics["roe"] = safe_get(info, "returnOnEquity") * 100 if safe_get(info, "returnOnEquity") else np.nan
    metrics["revenue_growth"] = compute_growth(income, "Total Revenue")

    return metrics

# Extract for all companies
metrics_list = []
for ticker, data in tqdm(lithium_data.items(), desc="Extracting metrics"):
    metrics = extract_mining_metrics(ticker, data)
    metrics_list.append(metrics)

df_metrics = pd.DataFrame(metrics_list)
df_metrics.set_index("ticker", inplace=True)

# Convert all numeric columns to proper numeric types
numeric_cols = ['market_cap', 'enterprise_value', 'current_price', 'revenue', 'gross_profit', 
                'operating_income', 'net_income', 'ebitda', 'gross_margin', 'operating_margin', 
                'net_margin', 'total_assets', 'current_assets', 'cash', 'total_debt', 
                'current_liabilities', 'stockholders_equity', 'current_ratio', 'debt_to_equity', 
                'operating_cf', 'capex', 'free_cash_flow', 'pe_ratio', 'pb_ratio', 'ev_ebitda', 
                'roe', 'revenue_growth']

for col in numeric_cols:
    if col in df_metrics.columns:
        df_metrics[col] = pd.to_numeric(df_metrics[col], errors='coerce')

print(f"Metrics extracted for {len(df_metrics)} companies!")
df_metrics[["name", "market_cap", "revenue", "net_margin", "pe_ratio"]].head(20)

Extracting metrics: 100%|██████████| 29/29 [00:00<00:00, 540.06it/s]


Metrics extracted for 29 companies!


,name,market_cap,revenue,net_margin,pe_ratio
ticker,,,,,
ALB,Albemarle Corporation,20920238080,5.142733e+09,-9.929117,NaN
LAC,Lithium Americas Corp,1386941568,0.000000e+00,NaN,NaN
LAC.TO,Lithium Americas Corp,1899836800,0.000000e+00,NaN,NaN
LTBR,Lithium Americas (Argentina),403994272,0.000000e+00,NaN,NaN
IONR,ioneer Ltd,311167552,0.000000e+00,NaN,NaN
LITM,Snow Lake Resources,53718284,0.000000e+00,NaN,NaN
FRHC,Freedom Holding Corp,7205797888,2.033455e+09,4.162866,5888.000000
SQM,Sociedad Química y Minera,20871555072,4.528761e+09,-8.928689,NaN
PLS.AX,Pilbara Minerals Limited,15202500608,8.902070e+08,-21.991065,NaN


## SECTION 3: DCF & NAV Valuation Models

Comprehensive valuation for producers and explorers.

In [11]:
# DCF Valuation for Producers
def dcf_lithium_miner(base_fcf, growth_rates, terminal_growth, wacc, debt, cash, shares, reserve_life_years=25):
    """
    Multi-stage DCF with mining-specific adjustments.
    growth_rates: list of growth rates for explicit forecast period
    """
    if pd.isna(base_fcf) or base_fcf <= 0:
        return None

    # Cap terminal growth by reserve life
    if reserve_life_years < 20:
        terminal_growth = min(terminal_growth, 0.01)  # 1% max for short-life assets

    # Project FCF
    fcf_projections = []
    current_fcf = base_fcf

    for growth in growth_rates:
        current_fcf = current_fcf * (1 + growth)
        fcf_projections.append(current_fcf)

    # Calculate PV of projected FCF
    pv_fcf = sum([fcf / ((1 + wacc) ** (i + 1)) for i, fcf in enumerate(fcf_projections)])

    # Terminal Value
    terminal_fcf = fcf_projections[-1] * (1 + terminal_growth)
    terminal_value = terminal_fcf / (wacc - terminal_growth)
    pv_terminal = terminal_value / ((1 + wacc) ** len(fcf_projections))

    # Enterprise Value
    enterprise_value = pv_fcf + pv_terminal

    # Equity Value
    equity_value = enterprise_value - debt + cash if pd.notna(debt) and pd.notna(cash) else enterprise_value

    # Per Share Value
    per_share_value = equity_value / shares if pd.notna(shares) and shares > 0 else np.nan

    return {
        "enterprise_value": enterprise_value,
        "equity_value": equity_value,
        "per_share_value": per_share_value,
        "pv_fcf": pv_fcf,
        "pv_terminal": pv_terminal
    }

# NAV Model for Explorers/Developers
def nav_valuation(project_npv, corporate_costs_pv, net_debt, shares, risk_discount=0.35):
    """Net Asset Value for pre-revenue companies."""
    if pd.isna(project_npv):
        return None

    # Apply development risk discount
    risked_npv = project_npv * (1 - risk_discount)

    # NAV = Risked NPV - Corporate Costs - Net Debt
    nav = risked_npv - corporate_costs_pv - net_debt if pd.notna(net_debt) else risked_npv - corporate_costs_pv

    # Per Share NAV
    nav_per_share = nav / shares if pd.notna(shares) and shares > 0 else np.nan

    return {
        "nav": nav,
        "nav_per_share": nav_per_share,
        "risked_npv": risked_npv
    }

print("Valuation models defined!")

Valuation models defined!


In [12]:
# Apply DCF to Producers
valuations = {}

for ticker in df_metrics[df_metrics["type"].isin(["Producer", "Integrated"])].index:
    row = df_metrics.loc[ticker]

    if pd.notna(row["free_cash_flow"]) and row["free_cash_flow"] > 0:
        valuation = dcf_lithium_miner(
            base_fcf=row["free_cash_flow"],
            growth_rates=[0.10, 0.08, 0.06, 0.04, 0.03],  # 5-year declining growth
            terminal_growth=0.025,  # 2.5% perpetual
            wacc=0.10,  # 10% discount rate
            debt=row["total_debt"],
            cash=row["cash"],
            shares=safe_get(lithium_data[ticker]["info"], "sharesOutstanding"),
            reserve_life_years=25
        )

        if valuation:
            valuations[ticker] = valuation
            print(f"{ticker:10s} | DCF Value: ${valuation['per_share_value']:.2f} | Current: ${row['current_price']:.2f}")

print(f"\nDCF valuations completed for {len(valuations)} companies")

ALB        | DCF Value: $80.40 | Current: $177.52

DCF valuations completed for 1 companies


## SECTION 4: Technical Analysis

RSI, MACD, Bollinger Bands, and multi-panel charts.

In [13]:
# Technical Indicators Calculation
def calculate_technical_indicators(price_data):
    """Calculate comprehensive technical indicators."""
    df = price_data[['Close']].copy()

    # Moving Averages
    df['SMA_50'] = df['Close'].rolling(50).mean()
    df['SMA_200'] = df['Close'].rolling(200).mean()
    df['EMA_12'] = df['Close'].ewm(span=12).mean()
    df['EMA_26'] = df['Close'].ewm(span=26).mean()

    # MACD
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_Signal'] = df['MACD'].ewm(span=9).mean()
    df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

    # RSI
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))

    # Bollinger Bands
    df['BB_Middle'] = df['Close'].rolling(20).mean()
    std = df['Close'].rolling(20).std()
    df['BB_Upper'] = df['BB_Middle'] + (2 * std)
    df['BB_Lower'] = df['BB_Middle'] - (2 * std)

    # Stochastic
    low_14 = df['Close'].rolling(14).min()
    high_14 = df['Close'].rolling(14).max()
    df['Stochastic_K'] = 100 * (df['Close'] - low_14) / (high_14 - low_14)
    df['Stochastic_D'] = df['Stochastic_K'].rolling(3).mean()

    return df

# Example: Calculate for a specific ticker
ticker_example = "ALB"
if ticker_example in lithium_data:
    ta_data = calculate_technical_indicators(lithium_data[ticker_example]["price_history"])
    print(f"Technical indicators calculated for {ticker_example}")
    ta_data.tail()

Technical indicators calculated for ALB


In [14]:
# Create Multi-Panel Technical Chart
def create_technical_chart(ticker, data):
    """Create comprehensive technical analysis chart."""
    price_history = data["price_history"]
    ta_data = calculate_technical_indicators(price_history)

    # Create subplots
    fig = make_subplots(
        rows=5, cols=1,
        shared_xaxes=True,
        row_heights=[0.4, 0.15, 0.15, 0.15, 0.15],
        subplot_titles=(
            f'{data["name"]} - Price & Moving Averages',
            'Volume',
            'MACD',
            'RSI',
            'Stochastic Oscillator'
        ),
        vertical_spacing=0.05
    )

    # Panel 1: Price + MAs + Bollinger Bands
    fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['Close'], name='Price', line=dict(color='white', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['SMA_50'], name='SMA 50', line=dict(color='orange', dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['SMA_200'], name='SMA 200', line=dict(color='blue', dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['BB_Upper'], name='BB Upper', line=dict(color='gray', dash='dot')), row=1, col=1)
    fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['BB_Lower'], name='BB Lower', line=dict(color='gray', dash='dot'), fill='tonexty'), row=1, col=1)

    # Panel 2: Volume
    fig.add_trace(go.Bar(x=price_history.index, y=price_history['Volume'], name='Volume', marker=dict(color='cyan')), row=2, col=1)

    # Panel 3: MACD
    fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['MACD'], name='MACD', line=dict(color='blue')), row=3, col=1)
    fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['MACD_Signal'], name='Signal', line=dict(color='red')), row=3, col=1)
    fig.add_trace(go.Bar(x=ta_data.index, y=ta_data['MACD_Hist'], name='Histogram', marker=dict(color='green')), row=3, col=1)

    # Panel 4: RSI
    fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['RSI'], name='RSI', line=dict(color='purple')), row=4, col=1)
    fig.add_hline(y=70, line_dash="dash", line_color="red", row=4, col=1)
    fig.add_hline(y=30, line_dash="dash", line_color="green", row=4, col=1)

    # Panel 5: Stochastic
    fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['Stochastic_K'], name='%K', line=dict(color='orange')), row=5, col=1)
    fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['Stochastic_D'], name='%D', line=dict(color='red')), row=5, col=1)

    fig.update_layout(height=1000, showlegend=True, template='plotly_dark')
    fig.update_xaxes(showgrid=False)

    return fig

# Example
fig_tech = create_technical_chart("ALB", lithium_data["ALB"])
fig_tech.show()

## SECTION 5: Geospatial Visualization

Interactive world maps with mine locations and production capacity.

In [15]:
# Load mine location data
with open('lithium_projects.json', 'r') as f:
    projects_data = json.load(f)

df_projects = pd.DataFrame(projects_data).T
print(f"Loaded {len(df_projects)} lithium projects")
df_projects.head()

Loaded 20 lithium projects


,company,ticker,project_name,location,latitude,longitude,production_capacity_kt,resource_type,status
ALB_Silver_Peak,ALB,ALB,Silver Peak,"Nevada, USA",37.9966,-117.9504,5.0,Brine,Producing
SQM_Atacama,SQM,SQM,Atacama Salar,"Antofagasta Region, Chile",-23.5954,-68.2623,180.0,Brine,Producing
PLS_Pilgangoora,Pilbara Minerals,PLS.AX,Pilgangoora,"Pilbara, Western Australia",-21.2531,118.6833,560.0,Hard Rock Spodumene,Producing
IGO_Greenbushes,IGO/Tianqi/ALB,IGO.AX,Greenbushes,Western Australia,-33.8548,116.0598,1300.0,Hard Rock Spodumene,Producing
MIN_Mt_Marion,Mineral Resources,MIN.AX,Mt Marion,Western Australia,-30.9742,119.6542,450.0,Hard Rock Spodumene,Producing


In [16]:
# Create Global Mine Map
def create_global_mine_map(df_projects):
    """Create interactive world map with mine locations."""

    # Convert production capacity to numeric to avoid dtype errors
    df_projects = df_projects.copy()
    df_projects['production_capacity_kt'] = pd.to_numeric(df_projects['production_capacity_kt'], errors='coerce')

    fig = px.scatter_geo(
        df_projects,
        lat='latitude',
        lon='longitude',
        size='production_capacity_kt',
        color='company',
        hover_name='project_name',
        hover_data={
            'status': True,
            'resource_type': True,
            'production_capacity_kt': True,
            'latitude': False,
            'longitude': False
        },
        projection='natural earth',
        title='🌍 Global Lithium Mining Operations by Company',
        template='plotly_dark',
        size_max=50
    )

    fig.update_geos(
        showland=True,
        landcolor='rgb(25, 25, 25)',
        coastlinecolor='rgb(100, 100, 100)',
        showcountries=True,
        countrycolor='rgb(80, 80, 80)',
        showlakes=True,
        lakecolor='rgb(20, 40, 60)',
        bgcolor='rgb(10, 10, 10)'
    )

    fig.update_layout(height=700)

    return fig

fig_map = create_global_mine_map(df_projects)
fig_map.show()

In [17]:
# Regional Production Analysis
# Convert to numeric first
df_projects_numeric = df_projects.copy()
df_projects_numeric['production_capacity_kt'] = pd.to_numeric(df_projects_numeric['production_capacity_kt'], errors='coerce')

regional_production = df_projects_numeric.groupby('location')['production_capacity_kt'].sum().sort_values(ascending=False)

fig_regional = go.Figure(data=[
    go.Bar(
        x=regional_production.values,
        y=regional_production.index,
        orientation='h',
        marker=dict(color='cyan')
    )
])

fig_regional.update_layout(
    title='Lithium Production Capacity by Region',
    xaxis_title='Production Capacity (kt/year)',
    yaxis_title='Region',
    template='plotly_dark',
    height=600
)

fig_regional.show()

## SECTION 6: Cost Curve Analysis

Production cost positioning across the industry.

In [18]:
# Load cost curve data
df_costs = pd.read_csv('lithium_cost_curve.csv')
print(f"Loaded cost data for {len(df_costs)} operations")
df_costs.head()

Loaded cost data for 20 operations


,ticker,company,project,production_kt,cash_cost_usd_per_ton,cost_quartile,year,resource_type
0,PLS.AX,Pilbara Minerals,Pilgangoora,560.0,450,Q1,2024,Hard Rock
1,MIN.AX,Mineral Resources,Mt Marion,450.0,550,Q1,2024,Hard Rock
2,MIN.AX,Mineral Resources,Wodgina,750.0,600,Q1,2024,Hard Rock
3,IGO.AX,IGO/Tianqi,Greenbushes,1300.0,650,Q1,2024,Hard Rock
4,SQM,SQM,Atacama,180.0,3500,Q2,2024,Brine


In [19]:
# Create Cost Curve Chart
def create_cost_curve(df_costs):
    """Create lithium industry cost curve."""

    # Sort by cost
    df_sorted = df_costs.sort_values('cash_cost_usd_per_ton').copy()
    df_sorted['cumulative_capacity'] = df_sorted['production_kt'].cumsum()

    fig = go.Figure()

    # Add cost curve
    fig.add_trace(go.Scatter(
        x=df_sorted['cumulative_capacity'],
        y=df_sorted['cash_cost_usd_per_ton'],
        mode='lines+markers',
        name='Cash Cost Curve',
        line=dict(shape='hv', width=2, color='orange'),
        marker=dict(size=10, color=df_sorted['production_kt'], colorscale='Viridis', showscale=True),
        text=df_sorted['company'] + ' - ' + df_sorted['project'],
        hovertemplate='<b>%{text}</b><br>Cumulative: %{x:.0f} kt<br>Cost: $%{y:.0f}/t<extra></extra>'
    ))

    # Add current lithium price
    current_price = 15000  # Example: $15k/ton
    fig.add_hline(
        y=current_price,
        line_dash='dash',
        line_color='green',
        annotation_text=f'Current Price: ${current_price}/t',
        annotation_position='right'
    )

    fig.update_layout(
        title='Global Lithium Production Cost Curve',
        xaxis_title='Cumulative Production Capacity (kt LCE)',
        yaxis_title='Cash Cost ($/ton LCE)',
        template='plotly_dark',
        height=600
    )

    return fig

fig_cost = create_cost_curve(df_costs)
fig_cost.show()

## SECTION 7: Composite Scoring & Ranking

Multi-factor investment scoring and top picks identification.

In [20]:
# Calculate Composite Investment Score
def calculate_composite_score(df):
    """Multi-factor scoring system (0-100)."""
    
    # Convert all numeric columns to proper numeric types
    numeric_cols = ['pe_ratio', 'pb_ratio', 'ev_ebitda', 'debt_to_equity', 'current_ratio', 
                    'cash', 'roe', 'operating_margin', 'revenue_growth', 'market_cap']
    
    df = df.copy()
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    scores_df = pd.DataFrame(index=df.index)

    # VALUATION (25 points)
    scores_df['pe_score'] = percentile_rank(df['pe_ratio'], higher_is_better=False)
    scores_df['pb_score'] = percentile_rank(df['pb_ratio'], higher_is_better=False)
    scores_df['ev_ebitda_score'] = percentile_rank(df['ev_ebitda'], higher_is_better=False)

    scores_df['valuation_score'] = (
        scores_df['pe_score'] * 0.35 +
        scores_df['pb_score'] * 0.35 +
        scores_df['ev_ebitda_score'] * 0.30
    ) * 0.25

    # FINANCIAL HEALTH (25 points)
    scores_df['debt_equity_score'] = percentile_rank(df['debt_to_equity'], higher_is_better=False)
    scores_df['current_ratio_score'] = percentile_rank(df['current_ratio'], higher_is_better=True)
    scores_df['cash_score'] = percentile_rank(df['cash'], higher_is_better=True)

    scores_df['financial_health_score'] = (
        scores_df['debt_equity_score'] * 0.40 +
        scores_df['current_ratio_score'] * 0.30 +
        scores_df['cash_score'] * 0.30
    ) * 0.25

    # PROFITABILITY (20 points)
    scores_df['roe_score'] = percentile_rank(df['roe'], higher_is_better=True)
    scores_df['margin_score'] = percentile_rank(df['operating_margin'], higher_is_better=True)

    scores_df['profitability_score'] = (
        scores_df['roe_score'] * 0.50 +
        scores_df['margin_score'] * 0.50
    ) * 0.20

    # GROWTH (15 points)
    scores_df['revenue_growth_score'] = percentile_rank(df['revenue_growth'], higher_is_better=True)
    scores_df['growth_score'] = scores_df['revenue_growth_score'] * 0.15

    # MARKET POSITION (15 points)
    scores_df['market_cap_score'] = percentile_rank(df['market_cap'], higher_is_better=True)
    scores_df['market_position_score'] = scores_df['market_cap_score'] * 0.15

    # TOTAL COMPOSITE SCORE
    scores_df['composite_score'] = (
        scores_df['valuation_score'] +
        scores_df['financial_health_score'] +
        scores_df['profitability_score'] +
        scores_df['growth_score'] +
        scores_df['market_position_score']
    )

    return scores_df

df_scores = calculate_composite_score(df_metrics)
print("Composite scores calculated!")
df_scores['composite_score'].describe()

Composite scores calculated!


count     5.000000
mean     60.531705
std       4.394404
min      55.528895
25%      57.167146
50%      60.421775
75%      63.197079
max      66.343630
Name: composite_score, dtype: float64

In [21]:
# Create Ranking Visualization
def create_ranking_chart(df_scores, df_metrics):
    """Horizontal bar chart of composite scores."""

    # Merge with company names
    ranking = df_scores[['composite_score']].join(df_metrics[['name']])
    ranking = ranking.sort_values('composite_score', ascending=True)

    # Color by score
    colors = ['#ff4757' if score < 40 else '#ffa502' if score < 60 else '#2ed573'
              for score in ranking['composite_score']]

    fig = go.Figure(data=[
        go.Bar(
            y=ranking.index + ' - ' + ranking['name'],
            x=ranking['composite_score'],
            orientation='h',
            marker=dict(color=colors),
            text=ranking['composite_score'].round(1),
            textposition='outside',
            hovertemplate='<b>%{y}</b><br>Score: %{x:.1f}/100<extra></extra>'
        )
    ])

    fig.update_layout(
        title='Lithium Miners - Composite Investment Score Ranking',
        xaxis_title='Composite Score (0-100)',
        yaxis_title='',
        height=max(600, len(ranking) * 25),
        template='plotly_dark',
        showlegend=False
    )

    return fig

fig_ranking = create_ranking_chart(df_scores, df_metrics)
fig_ranking.show()

In [22]:
# Spider Chart for Top Companies
def create_spider_chart(df_scores, df_metrics, top_n=5):
    """Radar chart comparing top companies."""

    # Get top companies
    top_companies = df_scores.nlargest(top_n, 'composite_score').index

    categories = ['Valuation', 'Financial Health', 'Profitability', 'Growth', 'Market Position']

    fig = go.Figure()

    for ticker in top_companies:
        scores = [
            df_scores.loc[ticker, 'valuation_score'],
            df_scores.loc[ticker, 'financial_health_score'],
            df_scores.loc[ticker, 'profitability_score'],
            df_scores.loc[ticker, 'growth_score'],
            df_scores.loc[ticker, 'market_position_score']
        ]

        fig.add_trace(go.Scatterpolar(
            r=scores,
            theta=categories,
            fill='toself',
            name=f"{ticker} - {df_metrics.loc[ticker, 'name']}"
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 25])),
        showlegend=True,
        title='Top Lithium Miners - Multi-Dimensional Comparison',
        template='plotly_dark',
        height=600
    )

    return fig

fig_spider = create_spider_chart(df_scores, df_metrics, top_n=5)
fig_spider.show()

## SECTION 8: Executive Summary & Recommendations

Top picks and sector-wide statistics.

In [23]:
# Generate Top Picks
top_picks = df_scores.nlargest(10, 'composite_score')[[
    'composite_score',
    'valuation_score',
    'financial_health_score',
    'profitability_score'
]].join(df_metrics[['name', 'type', 'market_cap', 'pe_ratio', 'roe']])

print("="*80)
print("TOP 10 LITHIUM MINING INVESTMENT OPPORTUNITIES")
print("="*80)
print("\nBased on Multi-Factor Composite Scoring (Valuation, Financial Health, Profitability, Growth, Market Position)\n")
print(top_picks.to_string())

# Export results
df_metrics.to_csv('lithium_miners_analysis_results.csv')
df_scores.to_csv('lithium_miners_investment_scores.csv')
print("\n✅ Results exported to CSV files!")

TOP 10 LITHIUM MINING INVESTMENT OPPORTUNITIES

Based on Multi-Factor Composite Scoring (Valuation, Financial Health, Profitability, Growth, Market Position)

           composite_score  valuation_score  financial_health_score  profitability_score                          name         type    market_cap     pe_ratio        roe
ticker                                                                                                                                                                   
RIO              66.343630        10.689655               14.233716            17.920259                     Rio Tinto  Diversified  162229616640    16.064144  16.399999
RIO.L            63.197079         8.577586               14.233716            17.920259  Rio Tinto - Lithium Division  Diversified  116397285376    15.915556  16.399999
002074.SZ        60.421775         9.310345                9.792465            14.870690                Tianqi Lithium   Integrated   68373946368    20.372972  1

In [24]:
# Summary Statistics
print("\\n" + "="*80)
print("LITHIUM MINING SECTOR SUMMARY")
print("="*80)

print(f"\\n📊 Coverage:")
print(f"   - Total companies analyzed: {len(df_metrics)}")
print(f"   - Integrated producers: {len(df_metrics[df_metrics['type']=='Integrated'])}")
print(f"   - Pure-play producers: {len(df_metrics[df_metrics['type']=='Producer'])}")
print(f"   - Developers: {len(df_metrics[df_metrics['type']=='Developer'])}")
print(f"   - Explorers: {len(df_metrics[df_metrics['type']=='Explorer'])}")

print(f"\\n💰 Market Overview:")
print(f"   - Total market cap: {format_currency(df_metrics['market_cap'].sum())}")
print(f"   - Median PE ratio: {df_metrics['pe_ratio'].median():.1f}x")
print(f"   - Median EV/EBITDA: {df_metrics['ev_ebitda'].median():.1f}x")

print(f"\\n📈 Profitability:")
print(f"   - Avg operating margin: {df_metrics['operating_margin'].mean():.1f}%")
print(f"   - Avg ROE: {df_metrics['roe'].mean():.1f}%")

print(f"\\n💪 Financial Health:")
print(f"   - Avg debt/equity: {df_metrics['debt_to_equity'].mean():.2f}x")
print(f"   - Avg current ratio: {df_metrics['current_ratio'].mean():.2f}x")

print("\\n✅ Analysis Complete!")

\n================================================================================
LITHIUM MINING SECTOR SUMMARY
\n📊 Coverage:
   - Total companies analyzed: 29
   - Integrated producers: 5
   - Pure-play producers: 4
   - Developers: 11
   - Explorers: 5
\n💰 Market Overview:
   - Total market cap: USD 593.47B
   - Median PE ratio: 20.4x
   - Median EV/EBITDA: -6.9x
\n📈 Profitability:
   - Avg operating margin: -36.8%
   - Avg ROE: -6.8%
\n💪 Financial Health:
   - Avg debt/equity: 0.50x
   - Avg current ratio: 7.30x
\n✅ Analysis Complete!


## SECTION 9: Data Quality & Limitations

Assessment of data completeness and reliability.

In [25]:
# Data Quality Assessment
data_quality = []

for ticker in df_metrics.index:
    quality = {
        'ticker': ticker,
        'name': df_metrics.loc[ticker, 'name'],
        'has_revenue': pd.notna(df_metrics.loc[ticker, 'revenue']),
        'has_financials': pd.notna(df_metrics.loc[ticker, 'ebitda']),
        'has_balance_sheet': pd.notna(df_metrics.loc[ticker, 'total_assets']),
        'has_cashflow': pd.notna(df_metrics.loc[ticker, 'operating_cf']),
    }

    # Calculate completeness score
    quality['completeness'] = sum([
        quality['has_revenue'],
        quality['has_financials'],
        quality['has_balance_sheet'],
        quality['has_cashflow']
    ]) * 25

    data_quality.append(quality)

df_quality = pd.DataFrame(data_quality)

print("\\nData Quality Summary:")
print(f"High quality (100%): {len(df_quality[df_quality['completeness']==100])} companies")
print(f"Good quality (75%): {len(df_quality[df_quality['completeness']==75])} companies")
print(f"Moderate quality (50%): {len(df_quality[df_quality['completeness']==50])} companies")
print(f"Low quality (<50%): {len(df_quality[df_quality['completeness']<50])} companies")

\nData Quality Summary:
High quality (100%): 16 companies
Good quality (75%): 8 companies
Moderate quality (50%): 5 companies
Low quality (<50%): 0 companies
